# Task E — One-page written justification

This is the compact report to accompany `taskD_review.png`. The calculations
below regenerate the evidence from the same documented rules as Part D,
then render the completed, data-specific justification. Keep the rendered
report to one page when exporting.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd()
WEEK4_DIR = HERE if (HERE / "assignments" / "data").exists() else HERE / "week4"
ASSIGNMENTS_DIR = WEEK4_DIR / "assignments"
DATA_DIR = ASSIGNMENTS_DIR / "data"
sys.path.insert(0, str(ASSIGNMENTS_DIR))
from tslib import PALETTE, style_defaults, save_fig, audit_index

style_defaults()

from IPython.display import Markdown
from tslib import window_report

raw = pd.read_csv(DATA_DIR / "sessions_hourly.csv")
iso = raw.ts.str.match(r"^\d{4}-\d{2}-\d{2} ")
parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
parsed.loc[iso] = pd.to_datetime(raw.loc[iso, "ts"], format="%Y-%m-%d %H:%M:%S")
parsed.loc[~iso] = pd.to_datetime(raw.loc[~iso, "ts"], format="%d/%m/%Y %H:%M")
clean = raw.assign(ts=parsed, device=raw.device.str.strip().str.title())
sentinel_n = int(clean.sessions.lt(0).sum())
clean = clean.loc[clean.sessions.ge(0)]
duplicate_n = int(clean.ts.duplicated().sum())
clean = clean.drop_duplicates("ts", keep="first").set_index("ts").sort_index()
full_hours = pd.date_range(clean.index.min().normalize(),
                           clean.index.max().normalize() + pd.Timedelta(hours=23), freq="h")
hourly = clean.sessions.astype(float).reindex(full_hours)
coverage = hourly.notna().resample("D").sum()
hour_profile = clean.sessions.groupby([clean.index.dayofweek, clean.index.hour]).median()
hourly_estimate = hourly.copy()
missing_index = hourly_estimate.index[hourly_estimate.isna()]
missing_keys = pd.MultiIndex.from_arrays([missing_index.dayofweek, missing_index.hour])
hourly_estimate.loc[missing_index] = [hour_profile.loc[key] for key in missing_keys]
observed_daily = hourly.resample("D").sum(min_count=1).asfreq("D")
daily = hourly_estimate.resample("D").sum(min_count=24).asfreq("D")
daily.name = "sessions_per_day"
missing_hours = int(hourly.isna().sum())
incomplete_days = int(coverage.lt(24).sum())
partial_month = daily.index.max().strftime("%B %Y")
partial_days = daily.index.max().day
month_days = daily.index.max().days_in_month

windows = window_report(daily, windows=(7, 28, 91), center=True)
weekly_removed = windows.set_index("window").loc[7, "variation_removed_pct"]

cat_raw = pd.read_csv(DATA_DIR / "revenue_catalogue.csv")
cat = cat_raw.drop_duplicates("sku", keep="first")
neg = cat.net_revenue_pkr_m.lt(0)
negative_count = int(neg.sum())
negative_total = float(cat.loc[neg, "net_revenue_pkr_m"].sum())
positive = cat.loc[~neg & cat.net_revenue_pkr_m.gt(0)]
subcat = positive.groupby("subcategory").net_revenue_pkr_m.sum()
positive_total = float(subcat.sum())
other_categories = int((subcat / positive_total * 100 < 1.0).sum())
other_total = float(subcat[subcat / positive_total * 100 < 1.0].sum())
sku_removed = len(cat_raw) - len(cat)

report = f"""
## E-commerce weekly review — method and design

**Panels.** The daily time-series panel answers whether traffic's longer trajectory changes after the weekly cycle is reduced. Position on a shared time axis is more precise than area; faint raw observations preserve spikes. The weekday-by-hour heatmap answers when traffic is high: a sequential single-hue scale represents non-negative magnitude. The treemap answers where positive net revenue is concentrated in the category hierarchy; area is useful for overview but imprecise, so direct labels and the adjacent sorted bars supply values. The bars answer exact subcategory ranking using length from a common zero baseline (position/length outrank angle/area). The panels have one job each; no dual y-axis is used.

**Time decisions.** ISO timestamps were parsed with `%Y-%m-%d %H:%M:%S`; slash timestamps with `%d/%m/%Y %H:%M`. Day-first is supported by slash dates whose first field exceeds 12 and by the April 2024–September 2025 range. The parsed, shuffled index was audited before sorting; one row per timestamp was kept (first source occurrence). Hourly sessions, orders, and revenue are flows, so daily sessions use a **sum**. The raw daily sum uses available hourly flows; missing hours are filled only in the separate daily trend estimate with the median for that weekday and hour. The 7-day **centred** rolling mean was fixed before plotting because it targets the weekly cycle; it reduces standard deviation by {weekly_removed:.1f}% in this extract. Centring is acceptable for the retrospective Monday report, not for live alerts. Missing hours remain `NaN` on the audited hourly index and flagged at daily resolution. {partial_month} is partial ({partial_days}/{month_days} days) and is marked, not compared with a complete month.

**Rows and periods.** From {len(raw):,} source rows, {sentinel_n:,} rows with negative session sentinels and {duplicate_n:,} duplicate timestamps were removed; the first remaining row at each timestamp was retained. Reindexing exposed {missing_hours:,} missing hourly bins and {incomplete_days:,} days with at least one missing hourly record. The weekday-hour imputation affects the estimate only; observed partial daily sums remain separate and flagged. The partial final month is retained at daily resolution, explicitly shaded, and excluded from complete-month comparisons. No plausible high traffic values were trimmed.

**Hierarchy.** One duplicated SKU row was removed by SKU key, retaining its first parent assignment ({sku_removed} row). {negative_count} negative net-revenue SKUs ({negative_total:.2f} PKR m combined) have no area and are disclosed rather than silently included. The hierarchy is aggregated from SKU to subcategory and parent category, then stops at subcategory: all {len(subcat)} subcategories are retained. `Other` contains {other_categories} subcategories below 1% each, worth {other_total:.2f} PKR m combined; no subcategories meet this threshold. Values are positive net revenue and all eight largest-subcategory shares are checked against the provided reference.

**Ethics.** The 7-day window and full available start date were fixed before inspecting the trend. The other window reports and the start/window sensitivity counts in Part B disclose how the framing affects the conclusion.
"""
display(Markdown(report))


## E-commerce weekly review — method and design

**Panels.** The daily time-series panel answers whether traffic's longer trajectory changes after the weekly cycle is reduced. Position on a shared time axis is more precise than area; faint raw observations preserve spikes. The weekday-by-hour heatmap answers when traffic is high: a sequential single-hue scale represents non-negative magnitude. The treemap answers where positive net revenue is concentrated in the category hierarchy; area is useful for overview but imprecise, so direct labels and the adjacent sorted bars supply values. The bars answer exact subcategory ranking using length from a common zero baseline (position/length outrank angle/area). The panels have one job each; no dual y-axis is used.

**Time decisions.** ISO timestamps were parsed with `%Y-%m-%d %H:%M:%S`; slash timestamps with `%d/%m/%Y %H:%M`. Day-first is supported by slash dates whose first field exceeds 12 and by the April 2024–September 2025 range. The parsed, shuffled index was audited before sorting; one row per timestamp was kept (first source occurrence). Hourly sessions, orders, and revenue are flows, so daily sessions use a **sum**. The raw daily sum uses available hourly flows; missing hours are filled only in the separate daily trend estimate with the median for that weekday and hour. The 7-day **centred** rolling mean was fixed before plotting because it targets the weekly cycle; it reduces standard deviation by 32.4% in this extract. Centring is acceptable for the retrospective Monday report, not for live alerts. Missing hours remain `NaN` on the audited hourly index and flagged at daily resolution. September 2025 is partial (11/30 days) and is marked, not compared with a complete month.

**Rows and periods.** From 12,670 source rows, 290 rows with negative session sentinels and 283 duplicate timestamps were removed; the first remaining row at each timestamp was retained. Reindexing exposed 599 missing hourly bins and 235 days with at least one missing hourly record. The weekday-hour imputation affects the estimate only; observed partial daily sums remain separate and flagged. The partial final month is retained at daily resolution, explicitly shaded, and excluded from complete-month comparisons. No plausible high traffic values were trimmed.

**Hierarchy.** One duplicated SKU row was removed by SKU key, retaining its first parent assignment (1 row). 7 negative net-revenue SKUs (-1.07 PKR m combined) have no area and are disclosed rather than silently included. The hierarchy is aggregated from SKU to subcategory and parent category, then stops at subcategory: all 8 subcategories are retained. `Other` contains 0 subcategories below 1% each, worth 0.00 PKR m combined; no subcategories meet this threshold. Values are positive net revenue and all eight largest-subcategory shares are checked against the provided reference.

**Ethics.** The 7-day window and full available start date were fixed before inspecting the trend. The other window reports and the start/window sensitivity counts in Part B disclose how the framing affects the conclusion.


Before submission, export/print the rendered report and verify it fits on
one page. The numerical counts are computed from the data, not hand-entered.
The report distinguishes row removal from missing time, and it states what
the `Other` bucket contains (including the zero-category case).